# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [25]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import gradio as gr # oh yeah!
import google.generativeai
import anthropic
from io import BytesIO
import re

# set up environment

load_dotenv()
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AIzaSyD7


In [18]:
# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them

openai = OpenAI()

claude = anthropic.Anthropic()

google.generativeai.configure()

In [19]:
system_message = """
You are a helpful assistant. You will be given a task and you should respond with the best possible answer. Ensure that your response is clear, concise, and relevant to the task at hand. If you are unsure about something, ask for clarification. 
Please assume I don't know anything about the task and provide a detailed explanation."""
system_message += "Ensure you don't give any answers regarding anything medical or that could be construed as medical advice. Also legal and fashion advice is off-limits."
system_message += "If you are unsure about something, say you don't know."

In [20]:
from pydub import AudioSegment
from pydub.playback import play

def talker(message):
    response = openai.audio.speech.create(
      model="tts-1",
      voice="onyx",    # Also, try replacing onyx with alloy
      input=message
    )
    
    audio_stream = BytesIO(response.content)
    audio = AudioSegment.from_file(audio_stream, format="mp3")
    play(audio)

In [24]:
talker("Hello, I am your assistant. How can I help you today?")

I0000 00:00:1749473302.334162 2257349 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1749473304.460083 2257349 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1749473304.607083 2257349 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers
Input #0, wav, from '/var/folders/57/166gcgt12qx_913_f14193p40000gp/T/tmpt1c28cus.wav':
  Duration: 00:00:03.17, bitrate: 384 kb/s
  Stream #0:0: Audio: pcm_s16le ([1][0][0][0] / 0x0001), 24000 Hz, 1 channels, s16, 384 kb/s


In [27]:
gemini = google.generativeai.GenerativeModel(
    model_name='gemini-2.0-flash',
    system_instruction=system_message
)

def stream_gemini(prompt):
    
    result = gemini.generate_content(prompt, stream=True)
    
    response = ""
    for chunk in result:
        response += chunk.text or ""
        yield response

def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = openai.chat.completions.create(
        model='gpt-4o-mini',
        messages=messages,
        stream=True
    )
    
    buffer = ""
    full_result = ""
    sentence_end_re = re.compile(r'([.!?])(\s|$)')

    for chunk in stream:
        content = chunk.choices[0].delta.content
        if content:
            full_result += content
            buffer += content
            yield full_result  # <- Still yielding the current cumulative result
            
            # Look for full sentences
            while True:
                match = sentence_end_re.search(buffer)
                if not match:
                    break
                end_idx = match.end()
                sentence = buffer[:end_idx].strip()
                buffer = buffer[end_idx:].lstrip()
                if sentence:
                    talker(sentence)

    # Speak any remaining buffer
    if buffer.strip():
        talker(buffer.strip())

def stream_claude(prompt):
    result = claude.messages.stream(
        model="claude-3-haiku-20240307",
        max_tokens=1000,
        temperature=0.7,
        system=system_message,
        messages=[
            {"role": "user", "content": prompt},
        ],
    )
    response = ""
    with result as stream:
        for text in stream.text_stream:
            response += text or ""
            yield response

def stream_model(prompt, model):
    if model=="GPT":
        result = stream_gpt(prompt)
    elif model=="Claude":
        result = stream_claude(prompt)
    elif model=="Gemini":
        result = stream_gemini(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [28]:
view = gr.Interface(
    fn=stream_model,
    inputs=[gr.Textbox(label="Your question:"), gr.Dropdown(["GPT", "Claude", "Gemini"], label="Select model", value="GPT")],
    outputs=[gr.Markdown(label="Response:")],
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7864

To create a public link, set `share=True` in `launch()`.


Input #0, wav, from '/var/folders/57/166gcgt12qx_913_f14193p40000gp/T/tmpm2uwr21y.wav':
  Duration: 00:00:00.48, bitrate: 384 kb/s
  Stream #0:0: Audio: pcm_s16le ([1][0][0][0] / 0x0001), 24000 Hz, 1 channels, s16, 384 kb/s


Input #0, wav, from '/var/folders/57/166gcgt12qx_913_f14193p40000gp/T/tmp3ksnk_1h.wav':
  Duration: 00:00:01.68, bitrate: 384 kb/s
  Stream #0:0: Audio: pcm_s16le ([1][0][0][0] / 0x0001), 24000 Hz, 1 channels, s16, 384 kb/s


Input #0, wav, from '/var/folders/57/166gcgt12qx_913_f14193p40000gp/T/tmprqbkwbun.wav':
  Duration: 00:00:02.83, bitrate: 384 kb/s
  Stream #0:0: Audio: pcm_s16le ([1][0][0][0] / 0x0001), 24000 Hz, 1 channels, s16, 384 kb/s


Input #0, wav, from '/var/folders/57/166gcgt12qx_913_f14193p40000gp/T/tmpmvl2w9_m.wav':
  Duration: 00:00:09.10, bitrate: 384 kb/s
  Stream #0:0: Audio: pcm_s16le ([1][0][0][0] / 0x0001), 24000 Hz, 1 channels, s16, 384 kb/s


Input #0, wav, from '/var/folders/57/166gcgt12qx_913_f14193p40000gp/T/tmpwk4fs79k.wav':
  Duration: 00:00:04.20, bitrate: 384 kb/s
  Stream #0:0: Audio: pcm_s16le ([1][0][0][0] / 0x0001), 24000 Hz, 1 channels, s16, 384 kb/s


Input #0, wav, from '/var/folders/57/166gcgt12qx_913_f14193p40000gp/T/tmpoij8q5z2.wav':
  Duration: 00:00:03.89, bitrate: 384 kb/s
  Stream #0:0: Audio: pcm_s16le ([1][0][0][0] / 0x0001), 24000 Hz, 1 channels, s16, 384 kb/s


Input #0, wav, from '/var/folders/57/166gcgt12qx_913_f14193p40000gp/T/tmpcuki7375.wav':
  Duration: 00:00:09.02, bitrate: 384 kb/s
  Stream #0:0: Audio: pcm_s16le ([1][0][0][0] / 0x0001), 24000 Hz, 1 channels, s16, 384 kb/s
